# FoodHub Order Analysis
## Exploratory Data Analysis — NYC Food Delivery Market

**Author:** Nikhil Patel  
**Date:** March 2026  
**Tools:** Python · Pandas · Matplotlib · Seaborn  

---

### Project Overview
FoodHub is a food aggregator company that connects customers in New York City with hundreds of restaurants through a single mobile app. The company earns revenue by collecting a fixed margin on each delivery order.

This project performs a comprehensive Exploratory Data Analysis (EDA) on **1,898 orders** to uncover demand patterns, delivery performance gaps, customer satisfaction insights, and revenue growth opportunities.

### Key Findings at a Glance
| Metric | Value |
|--------|-------|
| Total Orders Analyzed | 1,898 |
| Net Commission Revenue | \$6,166 |
| Avg Customer Rating | 4.34★ |
| Weekend Order Share | 71.2% |
| Unrated Orders | 38.8% |
| Orders >60 Min Delivery | 10.5% |


---
### Context

The number of restaurants in New York is increasing day by day. Lots of students and busy professionals rely on those restaurants due to their hectic lifestyles. Online food delivery service is a great option for them. It provides them with good food from their favorite restaurants. A food aggregator company **FoodHub** offers access to multiple restaurants through a single smartphone app.

The app allows the restaurants to receive a direct online order from a customer. The app assigns a delivery person from the company to pick up the order after it is confirmed by the restaurant. The delivery person then uses the map to reach the restaurant and waits for the food package. Once the food package is handed over to the delivery person, they confirm the pick-up in the app and start heading to the customer's location to deliver the package. The delivery person confirms the drop-off in the app after the food is delivered to the customer. The customer can rate the order in the app. The food aggregator earns money by collecting a fixed margin of the delivery order from the restaurants.

**Objective:** The food aggregator company has stored the data of the different orders made by the registered customers in their online portal. They want to analyze the data to get a fair idea about the demand of different restaurants which will help the company to enhance the business and customer experience.


### Import Required Libraries

In [ ]:
# Import libraries for data manipulation
import numpy as np
import pandas as pd

# Import libraries for data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visualization settings
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")


### Understanding the Structure of the Data

In [ ]:
# Read the dataset
df = pd.read_csv('foodhub_orders.csv')

# Display first 5 rows
df.head()


In [ ]:
# Display last 5 rows
df.tail()


### **Question 1:** How many rows and columns are present in the data?

In [ ]:
# Check the shape of the dataset
print(f"Dataset Shape: {df.shape}")
print(f"Number of Rows (Orders): {df.shape[0]}")
print(f"Number of Columns (Features): {df.shape[1]}")
print(f"\nColumn Names:\n{list(df.columns)}")


#### Observations:
- The dataset contains **1,898 rows** (orders) and **9 columns** (features).
- Each row represents a single food delivery order placed through the FoodHub app.
- The 9 features capture: order_id, customer_id, restaurant_name, cuisine_type, cost_of_the_order, day_of_the_week, rating, food_preparation_time, and delivery_time.


### **Question 2:** What are the datatypes of the different columns in the dataset?

In [ ]:
# Check datatypes and non-null counts
df.info()


In [ ]:
# Detailed dtype summary
print("\nData Types Summary:")
print(df.dtypes)


#### Observations:
- **Numerical columns (5):** order_id (int64), customer_id (int64), cost_of_the_order (float64), food_preparation_time (int64), delivery_time (int64)
- **Categorical columns (4):** restaurant_name (object), cuisine_type (object), day_of_the_week (object), rating (object)
- The `rating` column is stored as **object** type instead of numeric — this is because it contains the string `"Not given"` for unrated orders. This will need to be handled during analysis.
- All columns have **1,898 non-null entries** — no structural missing values.


### **Question 3:** Are there any missing values in the data? If yes, treat them using an appropriate method.

In [ ]:
# Check for missing values
print("Missing Values per Column:")
print(df.isnull().sum())
print(f"\nTotal Missing Values: {df.isnull().sum().sum()}")


In [ ]:
# Check for 'Not given' in rating column (implicit missing data)
not_given_count = (df['rating'] == 'Not given').sum()
print(f"\nOrders with 'Not given' rating: {not_given_count} ({not_given_count/len(df)*100:.1f}%)")
print(f"Orders with actual rating: {len(df) - not_given_count} ({(len(df) - not_given_count)/len(df)*100:.1f}%)")


#### Observations:
- There are **no null/NaN values** in any column.
- However, the `rating` column contains **736 entries (38.8%)** with the value `"Not given"` — these represent orders that customers chose not to rate.
- This is not technically missing data but represents a significant **feedback gap** that limits the company's ability to monitor service quality at scale.
- For analysis purposes, we will filter these out when analyzing rating distributions and create a separate `df_rated` subset.


### **Question 4:** Check the statistical summary of the data. What is the minimum, average, and maximum time it takes for food to be prepared once an order is placed?

In [ ]:
# Statistical summary of all numerical columns
df.describe().round(2)


In [ ]:
# Detailed food preparation time analysis
print("=" * 45)
print("  FOOD PREPARATION TIME STATISTICS")
print("=" * 45)
print(f"  Minimum:        {df['food_preparation_time'].min()} minutes")
print(f"  Mean (Average): {df['food_preparation_time'].mean():.2f} minutes")
print(f"  Median:         {df['food_preparation_time'].median():.1f} minutes")
print(f"  Maximum:        {df['food_preparation_time'].max()} minutes")
print(f"  Std Deviation:  {df['food_preparation_time'].std():.2f} minutes")
print("=" * 45)


#### Observations:
- **Food Preparation Time:** Min = 20 min, Average = 27.37 min, Max = 35 min — a relatively narrow 15-minute range.
- **Delivery Time:** Min = 15 min, Average = 24.16 min, Max = 33 min.
- **Order Cost:** Ranges from \$4.47 to \$35.41, with a mean of \$16.50 and median of \$14.14.
- The cost distribution is right-skewed (mean > median), indicating a cluster of lower-cost orders with some high-value outliers.
- Preparation times are remarkably consistent across restaurants, suggesting standardized kitchen workflows.


### **Question 5:** How many orders are not rated?

In [ ]:
# Count unrated vs rated orders
unrated = (df['rating'] == 'Not given').sum()
rated = len(df) - unrated

print(f"Unrated Orders: {unrated} ({unrated/len(df)*100:.1f}%)")
print(f"Rated Orders:   {rated} ({rated/len(df)*100:.1f}%)")

# Visualization
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#27ae60', '#e74c3c']
bars = ax.bar(['Rated', 'Not Rated'], [rated, unrated], color=colors, edgecolor='white', width=0.5)

for bar, val in zip(bars, [rated, unrated]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 15,
            f'{val} ({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontweight='bold', fontsize=13)

ax.set_title('Order Rating Status', fontsize=16, fontweight='bold', pad=15)
ax.set_ylabel('Number of Orders', fontsize=12)
sns.despine()
plt.tight_layout()
plt.show()


#### Observations:
- **736 orders (38.8%)** are not rated — nearly 4 in 10 orders lack customer feedback.
- Only **1,162 orders (61.2%)** have ratings.
- This is a **critical feedback gap** that limits FoodHub's ability to monitor restaurant quality and customer satisfaction.
- **Recommendation:** Implement in-app post-delivery rating prompts with loyalty points or discount incentives to boost completion rates.


---
## Exploratory Data Analysis (EDA)

### Univariate Analysis

### **Question 6:** Explore all the variables and provide observations on their distributions.


In [ ]:
# 1. Distribution of Order Cost
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['cost_of_the_order'], bins=20, color='#3498db', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution of Order Cost', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Cost ($)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['cost_of_the_order'].mean(), color='red', linestyle='--', linewidth=2,
                label=f"Mean: ${df['cost_of_the_order'].mean():.2f}")
axes[0].axvline(df['cost_of_the_order'].median(), color='orange', linestyle='--', linewidth=2,
                label=f"Median: ${df['cost_of_the_order'].median():.2f}")
axes[0].legend()

sns.boxplot(y=df['cost_of_the_order'], ax=axes[1], color='#3498db', width=0.4)
axes[1].set_title('Boxplot of Order Cost', fontweight='bold', fontsize=14)
axes[1].set_ylabel('Cost ($)')

plt.tight_layout()
plt.show()


In [ ]:
# 2. Distribution of Food Preparation Time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['food_preparation_time'], bins=15, color='#e67e22', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution of Food Preparation Time', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Time (minutes)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['food_preparation_time'].mean(), color='red', linestyle='--', linewidth=2,
                label=f"Mean: {df['food_preparation_time'].mean():.1f} min")
axes[0].legend()

sns.boxplot(y=df['food_preparation_time'], ax=axes[1], color='#e67e22', width=0.4)
axes[1].set_title('Boxplot of Preparation Time', fontweight='bold', fontsize=14)
axes[1].set_ylabel('Time (minutes)')

plt.tight_layout()
plt.show()


In [ ]:
# 3. Distribution of Delivery Time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['delivery_time'], bins=15, color='#27ae60', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution of Delivery Time', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Time (minutes)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['delivery_time'].mean(), color='red', linestyle='--', linewidth=2,
                label=f"Mean: {df['delivery_time'].mean():.1f} min")
axes[0].legend()

sns.boxplot(y=df['delivery_time'], ax=axes[1], color='#27ae60', width=0.4)
axes[1].set_title('Boxplot of Delivery Time', fontweight='bold', fontsize=14)
axes[1].set_ylabel('Time (minutes)')

plt.tight_layout()
plt.show()


In [ ]:
# 4. Distribution of Cuisine Types
fig, ax = plt.subplots(figsize=(12, 6))
cuisine_counts = df['cuisine_type'].value_counts()
palette = sns.color_palette('viridis', len(cuisine_counts))

bars = ax.barh(cuisine_counts.index[::-1], cuisine_counts.values[::-1], color=palette[::-1], edgecolor='white')
for bar, val in zip(bars, cuisine_counts.values[::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2.,
            f'{val} ({val/len(df)*100:.1f}%)', va='center', fontweight='bold', fontsize=11)

ax.set_title('Orders by Cuisine Type', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Number of Orders', fontsize=12)
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# 5. Orders by Day of the Week
fig, ax = plt.subplots(figsize=(8, 6))
day_counts = df['day_of_the_week'].value_counts()
colors = ['#3498db', '#e67e22']

wedges, texts, autotexts = ax.pie(day_counts, labels=day_counts.index, autopct='%1.1f%%',
                                   colors=colors, startangle=90, textprops={'fontsize': 13},
                                   explode=(0.02, 0.02))
for autotext in autotexts:
    autotext.set_fontweight('bold')

ax.set_title('Orders by Day Type', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


In [ ]:
# 6. Rating Distribution (rated orders only)
df_rated = df[df['rating'] != 'Not given'].copy()
df_rated['rating'] = df_rated['rating'].astype(int)

fig, ax = plt.subplots(figsize=(8, 5))
rating_counts = df_rated['rating'].value_counts().sort_index()
colors_rating = ['#e74c3c', '#f39c12', '#27ae60']
bars = ax.bar(rating_counts.index, rating_counts.values, color=colors_rating, edgecolor='white', width=0.6)

for bar, val in zip(bars, rating_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10,
            f'{val} ({val/len(df_rated)*100:.1f}%)', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_title('Rating Distribution (Rated Orders Only)', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Rating', fontsize=12)
ax.set_ylabel('Number of Orders', fontsize=12)
ax.set_xticks([3, 4, 5])
sns.despine()
plt.tight_layout()
plt.show()


#### Observations:
- **Order Cost:** Right-skewed distribution — mean (\$16.50) > median (\$14.14). Most orders are in the \$5–\$20 range, with fewer high-value orders above \$25.
- **Food Preparation Time:** Nearly uniform distribution between 20–35 minutes. Restaurants maintain consistent preparation workflows.
- **Delivery Time:** Ranges from 15–33 minutes with a mean of ~24 minutes.
- **Cuisine Type:** American (30.8%) and Japanese (24.8%) dominate, together accounting for **55.6%** of all orders. Italian (15.7%), Chinese (11.3%), and Mexican (4.1%) follow.
- **Day Type:** Weekends drive **71.2%** of all orders — a 2.5× demand multiplier over weekdays.
- **Ratings:** Of rated orders, 50.6% are 5-star, 33.2% are 4-star, and 16.2% are 3-star. **83% of rated orders score 4+ stars**, indicating strong satisfaction when feedback is given.


### **Question 7:** Which are the top 5 restaurants in terms of the number of orders received?

In [ ]:
# Top 5 restaurants by order volume
top5_restaurants = df['restaurant_name'].value_counts().head(5)

fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette('Blues_d', 5)[::-1]
bars = ax.barh(top5_restaurants.index[::-1], top5_restaurants.values[::-1], color=colors, edgecolor='white')

for bar, val in zip(bars, top5_restaurants.values[::-1]):
    ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2.,
            f'{val} orders ({val/len(df)*100:.1f}%)', va='center', fontweight='bold', fontsize=11)

ax.set_title('Top 5 Restaurants by Order Volume', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Number of Orders', fontsize=12)
sns.despine()
plt.tight_layout()
plt.show()

print("\nTop 5 Restaurants:")
for i, (name, count) in enumerate(top5_restaurants.items(), 1):
    print(f"  {i}. {name}: {count} orders ({count/len(df)*100:.1f}%)")


#### Observations:
- **Shake Shack** leads with 219 orders (11.5% of total volume) — a critical revenue partner.
- **The Meatball Shop** (132 orders), **Blue Ribbon Sushi** (119), **Blue Ribbon Fried Chicken** (96), and **Parm** (68) round out the top 5.
- These 5 restaurants collectively account for ~33% of all orders, indicating moderate platform concentration.
- Shake Shack alone handles more orders than the 4th and 5th restaurants combined — a key partner to protect.


### **Question 8:** Which is the most popular cuisine on weekends?

In [ ]:
# Most popular cuisine on weekends
weekend_df = df[df['day_of_the_week'] == 'Weekend']
weekend_cuisine = weekend_df['cuisine_type'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette('Set2', len(weekend_cuisine))
bars = ax.bar(weekend_cuisine.index, weekend_cuisine.values, color=colors, edgecolor='white')

for bar, val in zip(bars, weekend_cuisine.values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
            f'{val}', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_title('Cuisine Popularity on Weekends', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Cuisine Type', fontsize=12)
ax.set_ylabel('Number of Orders', fontsize=12)
plt.xticks(rotation=45, ha='right')
sns.despine()
plt.tight_layout()
plt.show()

print(f"\nMost popular weekend cuisine: {weekend_cuisine.index[0]} ({weekend_cuisine.values[0]} orders)")


#### Observations:
- **American cuisine** is the most popular on weekends with ~415 orders, followed by Japanese (~335) and Italian (~207).
- These three cuisines should be prioritized in weekend push notifications and promotional banners to capture peak demand.
- Weekend demand patterns closely mirror overall demand patterns, but with amplified volume.


### **Question 9:** What percentage of the orders cost more than 20 dollars?

In [ ]:
# Orders costing more than $20
high_value = (df['cost_of_the_order'] > 20).sum()
pct_high = high_value / len(df) * 100

print(f"Orders costing > $20: {high_value} ({pct_high:.1f}%)")
print(f"Orders costing <= $20: {len(df) - high_value} ({100 - pct_high:.1f}%)")

# Visualization
fig, ax = plt.subplots(figsize=(8, 5))
categories = ['≤ $20', '> $20']
values = [len(df) - high_value, high_value]
colors = ['#3498db', '#e74c3c']

bars = ax.bar(categories, values, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 15,
            f'{val} ({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontweight='bold', fontsize=13)

ax.set_title('Order Cost Distribution: Above vs Below $20', fontsize=16, fontweight='bold', pad=15)
ax.set_ylabel('Number of Orders', fontsize=12)
sns.despine()
plt.tight_layout()
plt.show()


#### Observations:
- **29.2% of orders** (555 orders) cost more than \$20 — these are high-value orders.
- The remaining 70.8% (1,343 orders) are \$20 or below.
- Despite being only ~29% of volume, high-value orders generate a disproportionate share of commission revenue due to the higher 25% commission rate.
- **Revenue implication:** Promoting premium orders above \$20 is the single highest-leverage revenue growth action.


### **Question 10:** What is the mean order delivery time?

In [ ]:
# Mean delivery time
mean_delivery = df['delivery_time'].mean()
median_delivery = df['delivery_time'].median()

print(f"Mean Delivery Time:   {mean_delivery:.2f} minutes")
print(f"Median Delivery Time: {median_delivery:.1f} minutes")
print(f"Min Delivery Time:    {df['delivery_time'].min()} minutes")
print(f"Max Delivery Time:    {df['delivery_time'].max()} minutes")


#### Observations:
- The **mean delivery time is ~24.16 minutes** and the median is ~24 minutes.
- Delivery times range from 15 to 33 minutes.
- The mean and median are very close, suggesting a roughly symmetric distribution without extreme outliers.


### **Question 11:** The company has decided to give 20% discount vouchers to the top 3 most frequent customers. Find the IDs of these customers and the number of orders they have placed.

In [ ]:
# Top 3 most frequent customers
top_customers = df['customer_id'].value_counts().head(5)

print("Top 5 Most Frequent Customers:")
print("=" * 40)
for i, (cid, count) in enumerate(top_customers.items(), 1):
    voucher = " ← 20% Voucher" if i <= 3 else ""
    print(f"  {i}. Customer {cid}: {count} orders{voucher}")

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#27ae60' if i < 3 else '#bdc3c7' for i in range(5)]
bars = ax.bar([str(x) for x in top_customers.index], top_customers.values, color=colors, edgecolor='white')

for bar, val in zip(bars, top_customers.values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2,
            f'{val} orders', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_title('Top 5 Customers by Order Frequency', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Customer ID', fontsize=12)
ax.set_ylabel('Number of Orders', fontsize=12)
ax.legend(['Top 3 (Voucher Recipients)', 'Others'], loc='upper right')
sns.despine()
plt.tight_layout()
plt.show()


#### Observations:
- The top 3 most frequent customers should receive 20% discount vouchers as a loyalty reward.
- These power users represent a small fraction of the customer base but drive outsized order volume.
- A loyalty tier program anchored around these top customers can drive repeat spend and word-of-mouth referrals.


---
## Multivariate Analysis

### **Question 12:** Perform a multivariate analysis to explore relationships between the important variables in the dataset.


In [ ]:
# Create a numeric version of rating for correlation analysis
df_analysis = df.copy()
df_analysis['rating_numeric'] = df_analysis['rating'].apply(lambda x: np.nan if x == 'Not given' else int(x))
df_analysis['total_time'] = df_analysis['food_preparation_time'] + df_analysis['delivery_time']

# Correlation heatmap (numerical variables)
fig, ax = plt.subplots(figsize=(8, 6))
corr_cols = ['cost_of_the_order', 'food_preparation_time', 'delivery_time', 'rating_numeric', 'total_time']
corr_matrix = df_analysis[corr_cols].corr()

sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.3f',
            linewidths=1, linecolor='white', square=True, ax=ax,
            xticklabels=['Cost', 'Prep Time', 'Delivery Time', 'Rating', 'Total Time'],
            yticklabels=['Cost', 'Prep Time', 'Delivery Time', 'Rating', 'Total Time'])
ax.set_title('Correlation Heatmap of Key Variables', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


In [ ]:
# Cuisine type vs. Average cost
fig, ax = plt.subplots(figsize=(12, 5))
cuisine_cost = df.groupby('cuisine_type')['cost_of_the_order'].mean().sort_values(ascending=False)

bars = ax.bar(cuisine_cost.index, cuisine_cost.values, color=sns.color_palette('viridis', len(cuisine_cost)), edgecolor='white')
for bar, val in zip(bars, cuisine_cost.values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2,
            f'${val:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_title('Average Order Cost by Cuisine Type', fontsize=16, fontweight='bold', pad=15)
ax.set_ylabel('Average Cost ($)', fontsize=12)
plt.xticks(rotation=45, ha='right')
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# Delivery time comparison: Weekday vs Weekend
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
sns.boxplot(x='day_of_the_week', y='delivery_time', data=df, ax=axes[0],
            palette=['#e74c3c', '#3498db'], order=['Weekday', 'Weekend'])
axes[0].set_title('Delivery Time: Weekday vs Weekend', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Day Type', fontsize=12)
axes[0].set_ylabel('Delivery Time (min)', fontsize=12)

# Mean comparison bar chart
day_delivery = df.groupby('day_of_the_week')['delivery_time'].mean()
bars = axes[1].bar(['Weekday', 'Weekend'], [day_delivery['Weekday'], day_delivery['Weekend']],
                    color=['#e74c3c', '#3498db'], edgecolor='white', width=0.5)
for bar, val in zip(bars, [day_delivery['Weekday'], day_delivery['Weekend']]):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                 f'{val:.1f} min', ha='center', va='bottom', fontweight='bold', fontsize=13)
axes[1].set_title('Average Delivery Time by Day Type', fontweight='bold', fontsize=14)
axes[1].set_ylabel('Avg Delivery Time (min)', fontsize=12)

sns.despine()
plt.tight_layout()
plt.show()

pct_diff = ((day_delivery['Weekday'] - day_delivery['Weekend']) / day_delivery['Weekend']) * 100
print(f"\nWeekday avg: {day_delivery['Weekday']:.1f} min | Weekend avg: {day_delivery['Weekend']:.1f} min")
print(f"Weekday delivery is {pct_diff:.0f}% slower than weekends")


In [ ]:
# Rating by Cuisine Type
fig, ax = plt.subplots(figsize=(12, 5))
cuisine_rating = df_analysis.groupby('cuisine_type')['rating_numeric'].mean().sort_values(ascending=True).dropna()

bars = ax.barh(cuisine_rating.index, cuisine_rating.values, color=sns.color_palette('YlOrRd', len(cuisine_rating)), edgecolor='white')
for bar, val in zip(bars, cuisine_rating.values):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2.,
            f'{val:.2f}★', va='center', fontweight='bold', fontsize=11)

ax.set_title('Average Customer Rating by Cuisine Type', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Average Rating', fontsize=12)
ax.set_xlim(3.5, 5.2)
sns.despine()
plt.tight_layout()
plt.show()


#### Observations:
- **Weak correlations** between most numerical variables — cost, preparation time, and delivery time are largely independent.
- **Weekday delivery is ~26% slower** than weekends (28.3 min vs 22.5 min) — a significant logistics gap.
- **Spanish (4.83★), Thai (4.67★), and Indian (4.54★)** cuisines lead customer satisfaction despite low order volumes — signaling untapped growth potential.
- **American and Japanese** cuisines have the highest average order costs and drive the most revenue.
- The delivery time gap on weekdays represents a clear operational improvement opportunity.


### **Question 13:** The company wants to provide a promotional offer in the advertisement of the restaurants. The condition to get the offer is that the restaurants must have a rating count of more than 50 and the average rating should be greater than 4. Find the restaurants fulfilling the criteria.

In [ ]:
# Filter restaurants eligible for promotional offers
# Criteria: rating count > 50 AND average rating > 4.0
restaurant_stats = df_analysis.groupby('restaurant_name').agg(
    avg_rating=('rating_numeric', 'mean'),
    rating_count=('rating_numeric', 'count'),
    total_orders=('order_id', 'count'),
    avg_cost=('cost_of_the_order', 'mean')
).reset_index()

# Filter: rated count > 50 and avg rating > 4
# Need to count only rated orders
rated_counts = df_rated.groupby('restaurant_name')['rating'].count().reset_index()
rated_counts.columns = ['restaurant_name', 'rated_order_count']
rated_avg = df_rated.groupby('restaurant_name')['rating'].apply(lambda x: x.astype(int).mean()).reset_index()
rated_avg.columns = ['restaurant_name', 'avg_rating']

promo_df = rated_counts.merge(rated_avg, on='restaurant_name')
promo_eligible = promo_df[(promo_df['rated_order_count'] > 50) & (promo_df['avg_rating'] > 4)]
promo_eligible = promo_eligible.sort_values('avg_rating', ascending=False)

print("Restaurants Eligible for Promotional Offer:")
print("Criteria: Rating Count > 50 AND Average Rating > 4.0")
print("=" * 60)
for _, row in promo_eligible.iterrows():
    print(f"  {row['restaurant_name']}")
    print(f"    Avg Rating: {row['avg_rating']:.2f}★ | Rated Orders: {int(row['rated_order_count'])}")
    print()


#### Observations:
- **4 restaurants** meet the promotional criteria (rating count > 50 and avg rating > 4.0):
  - The Meatball Shop (4.51★, 84 ratings)
  - Blue Ribbon Fried Chicken (4.33★, 64 ratings)
  - Shake Shack (4.28★, 133 ratings)
  - Blue Ribbon Sushi (4.22★, 73 ratings)
- **Shake Shack** has the highest volume + strong rating — priority partner for homepage features.
- These restaurants represent quality leaders on the platform and should be prominently featured in promotional campaigns.


### **Question 14:** The company charges the restaurant 25% on the orders having cost greater than 20 dollars and 15% on the orders having cost greater than 5 dollars. Find the net revenue generated by the company across all orders.

In [ ]:
# Calculate commission revenue
def calculate_commission(cost):
    if cost > 20:
        return cost * 0.25
    elif cost > 5:
        return cost * 0.15
    else:
        return 0

df['commission'] = df['cost_of_the_order'].apply(calculate_commission)

# Revenue breakdown
total_revenue = df['commission'].sum()
high_tier = df[df['cost_of_the_order'] > 20]['commission'].sum()
standard_tier = df[(df['cost_of_the_order'] > 5) & (df['cost_of_the_order'] <= 20)]['commission'].sum()
no_commission = df[df['cost_of_the_order'] <= 5]['commission'].sum()

print("=" * 50)
print("  REVENUE BREAKDOWN")
print("=" * 50)
print(f"  High-Tier (>$20 @ 25%):    ${high_tier:,.2f}  ({df[df['cost_of_the_order'] > 20].shape[0]} orders)")
print(f"  Standard ($5-$20 @ 15%):   ${standard_tier:,.2f}  ({df[(df['cost_of_the_order'] > 5) & (df['cost_of_the_order'] <= 20)].shape[0]} orders)")
print(f"  No Commission (≤$5):       ${no_commission:,.2f}  ({df[df['cost_of_the_order'] <= 5].shape[0]} orders)")
print(f"  {'─' * 46}")
print(f"  TOTAL NET REVENUE:         ${total_revenue:,.2f}")
print("=" * 50)

# Revenue by cuisine
cuisine_revenue = df.groupby('cuisine_type')['commission'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(cuisine_revenue.index, cuisine_revenue.values,
              color=sns.color_palette('Blues_d', len(cuisine_revenue))[::-1], edgecolor='white')
for bar, val in zip(bars, cuisine_revenue.values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,
            f'${val:,.0f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_title('Estimated Commission Revenue by Cuisine', fontsize=16, fontweight='bold', pad=15)
ax.set_ylabel('Revenue ($)', fontsize=12)
plt.xticks(rotation=45, ha='right')
sns.despine()
plt.tight_layout()
plt.show()


#### Observations:
- **Total net revenue: ~\$6,166** across 1,898 orders.
- **High-tier orders (>\$20)** generate ~\$3,689 in commission from only 555 orders (29.2% of volume but ~60% of revenue).
- **Standard tier (\$5-\$20)** generates ~\$2,478 from 1,262 orders.
- **American** cuisine leads revenue (~\$1,906), followed by Japanese (~\$1,533) and Italian (~\$979).
- **Key insight:** Promoting premium orders above the \$20 threshold is the single highest-leverage revenue growth action. Shifting just 5% more orders above \$20 could add ~\$150 in additional commission.


### **Question 15:** The company wants to analyze the total time required to deliver the food. What percentage of orders take more than 60 minutes to get delivered from the time the order is placed? (The food has to be prepared and then delivered.)

In [ ]:
# Calculate total fulfillment time
df['total_time'] = df['food_preparation_time'] + df['delivery_time']

over_60 = (df['total_time'] > 60).sum()
pct_over_60 = over_60 / len(df) * 100

print(f"Total Fulfillment Time Statistics:")
print(f"  Average: {df['total_time'].mean():.1f} minutes")
print(f"  Median:  {df['total_time'].median():.1f} minutes")
print(f"  Min:     {df['total_time'].min()} minutes")
print(f"  Max:     {df['total_time'].max()} minutes")
print(f"\nOrders taking > 60 minutes: {over_60} ({pct_over_60:.1f}%)")

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['total_time'], bins=25, color='#8e44ad', edgecolor='white', alpha=0.85)
ax.axvline(60, color='red', linestyle='--', linewidth=2, label=f'60 min threshold ({pct_over_60:.1f}% exceed)')
ax.axvline(df['total_time'].mean(), color='orange', linestyle='--', linewidth=2,
           label=f"Mean: {df['total_time'].mean():.1f} min")
ax.set_title('Distribution of Total Fulfillment Time (Prep + Delivery)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Total Time (minutes)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.legend(fontsize=11)
sns.despine()
plt.tight_layout()
plt.show()


#### Observations:
- The average total fulfillment time is **~51.5 minutes** (prep + delivery).
- **10.5% of orders** (approximately 200 orders) exceed the **60-minute threshold** — a customer satisfaction risk.
- Orders exceeding 60 minutes likely face higher cancellation and negative review rates.
- **Recommendation:** Flag orders approaching 55+ minutes for proactive customer communication and investigate root causes (slow prep vs. slow delivery).


### **Question 16:** The company wants to analyze the delivery time of the orders on weekdays and weekends. How does the mean delivery time vary during weekdays and weekends?

In [ ]:
# Delivery time analysis by day type
weekday_delivery = df[df['day_of_the_week'] == 'Weekday']['delivery_time']
weekend_delivery = df[df['day_of_the_week'] == 'Weekend']['delivery_time']

print("Delivery Time by Day Type:")
print("=" * 50)
print(f"  Weekday Mean: {weekday_delivery.mean():.1f} min (n={len(weekday_delivery)} orders)")
print(f"  Weekend Mean: {weekend_delivery.mean():.1f} min (n={len(weekend_delivery)} orders)")
print(f"  Difference:   {weekday_delivery.mean() - weekend_delivery.mean():.1f} min slower on weekdays")
print(f"  % Difference:  {((weekday_delivery.mean() - weekend_delivery.mean()) / weekend_delivery.mean()) * 100:.0f}% slower")

# Detailed visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram comparison
axes[0].hist(weekday_delivery, bins=15, alpha=0.7, color='#e74c3c', label=f'Weekday (μ={weekday_delivery.mean():.1f})', edgecolor='white')
axes[0].hist(weekend_delivery, bins=15, alpha=0.7, color='#3498db', label=f'Weekend (μ={weekend_delivery.mean():.1f})', edgecolor='white')
axes[0].set_title('Delivery Time Distribution by Day Type', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Delivery Time (minutes)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].legend(fontsize=11)

# Violin plot
sns.violinplot(x='day_of_the_week', y='delivery_time', data=df, ax=axes[1],
               palette=['#e74c3c', '#3498db'], order=['Weekday', 'Weekend'], inner='box')
axes[1].set_title('Delivery Time Violin Plot', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Day Type', fontsize=12)
axes[1].set_ylabel('Delivery Time (min)', fontsize=12)

sns.despine()
plt.tight_layout()
plt.show()


#### Observations:
- **Weekday mean delivery: ~28.3 minutes** vs **Weekend mean: ~22.5 minutes**.
- Weekdays are approximately **26% slower** — a significant logistics gap.
- Possible causes: traffic congestion during business hours, fewer drivers available on weekdays, or suboptimal routing during peak commute times.
- **Recommendation:** Audit weekday routing algorithms, increase driver incentives during weekday hours, and coordinate restaurant pickup times to reduce wait periods.


---
## Conclusions and Recommendations

### **Question 17:** What are your conclusions from the analysis? What recommendations would you like to share to help improve the business?


### Conclusions:

1. **Demand is heavily weekend-driven:** 71.2% of all orders occur on weekends, representing a 2.5× multiplier over weekdays. This has major implications for staffing, inventory, and marketing.

2. **Two cuisines dominate the market:** American (30.8%) and Japanese (24.8%) together account for 55.6% of all orders and \$3,439 in combined commission revenue. Shake Shack alone captures 11.5% of total volume.

3. **Weekday delivery performance lags significantly:** Average weekday delivery (28.3 min) is 26% slower than weekends (22.5 min) — this gap likely erodes customer satisfaction and repeat order rates during the work week.

4. **A critical feedback gap exists:** 38.8% of orders go unrated, severely limiting the company's ability to monitor quality and identify underperforming restaurants.

5. **High-value orders are the primary revenue driver:** Only 29.2% of orders exceed \$20, but these orders generate approximately 60% of total commission revenue due to the tiered commission structure.

6. **Untapped high-satisfaction cuisines exist:** Spanish (4.83★), Thai (4.67★), and Indian (4.54★) have the highest customer ratings but very low order volumes — a clear growth opportunity.

7. **10.5% of orders exceed 60 minutes end-to-end:** These long-wait orders represent a direct customer satisfaction and retention risk.


### Recommendations:

1. **Scale Weekend Operations:** Increase delivery staff and expand restaurant partnerships on Saturdays and Sundays to reduce wait times and capture additional unmet demand during peak periods.

2. **Fix Weekday Delivery Speed:** Audit routing algorithms, introduce driver incentives for weekday shifts, and coordinate restaurant pickup windows to close the 26% speed gap.

3. **Close the Ratings Gap:** Launch in-app post-delivery rating prompts with loyalty points or small discount coupons as completion incentives. Target reducing the unrated percentage from 38.8% to below 15%.

4. **Grow High-Satisfaction Cuisines:** Design targeted marketing campaigns for Spanish, Thai, and Indian restaurants. Their exceptional ratings indicate strong product-market fit that isn't yet matched by discovery and demand.

5. **Accelerate High-Value Orders:** Promote premium meal bundles, family packs, and curated restaurant collections to shift more orders above the \$20 threshold. Even a 5% shift adds ~\$150 in incremental commission.

6. **Activate Loyalty Programs:** Issue 20% discount vouchers to the top 5 most frequent customers. Feature the 4 promo-eligible restaurants (all rated 4.22–4.51★) prominently in the app to reward consistent quality.


---
*Analysis completed by Nikhil Patel | March 2026*